# Hypothesis Test: Are Newer Partners the Fraud Problem?

**Kestrel Home: Warranty Claim Fraud**

> *"I'm fairly sure it's the newer partners, but let's have the data say it."*
> Ritu Deshpande, Head of D2C Operations

| | |
|---|---|
| **H0** | Fraud is independent of partner tenure |
| **H1** | Claims from newer partners are more likely to be fraud |
| **Data** | 11,146 labelled claims, Apr 2025 to Jun 2026 (`train.csv` + `partners.csv`) |
| **Methods** | Fraud rates, chi-square test, logistic regression, partner-level test |

## Executive Summary

**Verdict: partly true.** Newer partners as a group are not the problem. After the 1 May 2026 auto-approval rule, **five recently onboarded outlets** account for most of the fraud.

| Finding | Evidence |
|---|---|
| Before May 2026, new partners were **cleaner** than established ones | 0.07% vs 1.25% fraud rate |
| After May 2026, new-partner claims are **far riskier** | 16.1% vs 0.17% (p < 0.001) |
| Over the full period, tenure does **not** predict fraud | Logistic β₁ = −0.04, p = 0.82 |
| New partners are **not** more likely to be fraudsters as a group | 17% vs 13% of partners with any fraud, p = 0.62 |
| **5 outlets** drive the effect | 29 of 35 new-partner frauds; without them, new partners' rate is 0.41% |

**Recommendation:** target specific outlets, not the new-partner cohort. 38 of 46 new partners are clean.

## 1. Setup

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

POLICY_CHANGE = pd.Timestamp("2026-05-01")   # auto-approval of claims < Rs 2,000 starts
COHORT_START = pd.Timestamp("2025-07-01")    # "new cohort" = onboarded in the last year

In [ ]:
DATA_DIR = next(p / "Data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "Data").is_dir())
find_file = lambda suffix: next(DATA_DIR.glob(f"*{suffix}"))

train = pd.read_csv(find_file("train.csv"), parse_dates=["submitted_at"])
partners = pd.read_csv(find_file("partners.csv"), parse_dates=["onboarded_date"])
partners["new_cohort"] = (partners["onboarded_date"] >= COHORT_START).astype(int)

# Keep first submission per claim, join partners
all_claims = (train.sort_values("submitted_at")
                   .drop_duplicates("claim_id", keep="first")
                   .merge(partners, on="partner_id", how="left", validate="many_to_one"))
all_claims["tenure_days"] = (all_claims["submitted_at"] - all_claims["onboarded_date"]).dt.days
all_claims["tenure_years"] = all_claims["tenure_days"] / 365
all_claims["new_partner"] = (all_claims["tenure_days"] < 365).astype(int)
all_claims["post_may"] = (all_claims["submitted_at"] >= POLICY_CHANGE).astype(int)
all_claims["group"] = all_claims["new_partner"].map({1: "New (< 1 yr)", 0: "Established"})
all_claims["period"] = all_claims["post_may"].map({0: "Before May 2026", 1: "After May 2026"})

# Drop open investigations (blank label)
claims = all_claims.dropna(subset=["is_fraud"]).copy()
claims["is_fraud"] = claims["is_fraud"].astype(int)

pre, post = claims[claims["post_may"] == 0], claims[claims["post_may"] == 1]
print(f"Labelled claims: {len(claims):,} | fraud: {claims['is_fraud'].sum()} ({claims['is_fraud'].mean():.2%})")
print(f"New-partner claims: {claims['new_partner'].sum():,} | partners in new cohort: {partners['new_cohort'].sum()}")

In [ ]:
def rate_table(df, by):
    """Claims, frauds and fraud rate (%) per group."""
    t = df.groupby(by, observed=True)["is_fraud"].agg(claims="size", frauds="sum", rate="mean")
    t["fraud_rate_%"] = (t.pop("rate") * 100).round(2)
    return t


def chi_square_2x2(df, label):
    """Chi-square and Fisher test of new_partner x is_fraud."""
    t = pd.crosstab(df["new_partner"], df["is_fraud"]).reindex(index=[1, 0], columns=[1, 0], fill_value=0)
    (a, b), (c, d) = t.values
    chi2, p, _, expected = stats.chi2_contingency(t)
    rate_new, rate_est = a / (a + b), c / (c + d)
    return {
        "Test": label,
        "New fraud %": round(rate_new * 100, 2),
        "Established fraud %": round(rate_est * 100, 2),
        "Relative risk": round(rate_new / rate_est, 2),
        "Chi2": round(chi2, 1),
        "p (chi2)": f"{p:.1e}",
        "p (Fisher)": f"{stats.fisher_exact(t)[1]:.1e}",
        "Min expected": round(expected.min(), 1),
    }


def logit_clustered(formula, df):
    """Logistic regression with standard errors clustered by partner."""
    return smf.logit(formula, df).fit(disp=0, cov_type="cluster",
                                      cov_kwds={"groups": pd.factorize(df["partner_id"])[0]})


def coef_table(model, label, terms):
    ci = np.exp(model.conf_int().loc[terms])
    return pd.DataFrame({
        "Model": label,
        "beta": model.params[terms].round(3),
        "Odds ratio": np.exp(model.params[terms]).round(2),
        "95% CI": [f"{lo:.2f} - {hi:.2f}" for lo, hi in ci.values],
        "p": model.pvalues[terms].map("{:.1e}".format),
    })

## 2. Fraud Rate: New vs Established

**Question:** Do claims from partners in their first year have a higher fraud rate?

In [ ]:
rates = rate_table(claims, ["period", "group"]).reindex(["Before May 2026", "After May 2026"], level=0)
overall = rate_table(claims, "group").assign(period="All data").set_index("period", append=True).swaplevel()
pd.concat([overall, rates])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_df = claims.groupby(["period", "group"])["is_fraud"].mean().mul(100).rename("rate").reset_index()
sns.barplot(data=plot_df, x="period", y="rate", hue="group", order=["Before May 2026", "After May 2026"], ax=axes[0])
for cont in axes[0].containers:
    axes[0].bar_label(cont, fmt="%.2f%%")
axes[0].set(title="Fraud rate by partner group", xlabel="", ylabel="Fraud rate (%)")

claims["tenure_band"] = pd.cut(claims["tenure_days"], [0, 90, 180, 365, 730, 10_000], right=False,
                               labels=["< 3 m", "3-6 m", "6-12 m", "1-2 y", "> 2 y"])
band = claims.groupby(["tenure_band", "period"], observed=True)["is_fraud"].mean().mul(100).rename("rate").reset_index()
sns.barplot(data=band, x="tenure_band", y="rate", hue="period", hue_order=["Before May 2026", "After May 2026"], ax=axes[1])
axes[1].set(title="Fraud rate by partner tenure", xlabel="Tenure at claim", ylabel="Fraud rate (%)")

plt.tight_layout(); plt.show()

**Result**
- Over all data, new partners look about **2× riskier** (2.18% vs 1.11%).
- Split by period, the picture reverses: **before May** new partners were cleaner (0.07% vs 1.25%); **after May** they are far riskier (16.1% vs 0.17%).

## 3. Chi-Square Test of Independence

**Question:** Is the difference in fraud rate statistically significant?
Fisher's exact test is shown as a check; it is preferred when the smallest expected count is below 5.

In [ ]:
pd.DataFrame([
    chi_square_2x2(claims, "All data"),
    chi_square_2x2(pre, "Before May 2026"),
    chi_square_2x2(post, "After May 2026"),
]).set_index("Test")

In [ ]:
band_table = pd.crosstab(claims["tenure_band"], claims["is_fraud"])
chi2, p, dof, _ = stats.chi2_contingency(band_table)
cramers_v = np.sqrt(chi2 / (band_table.values.sum() * (min(band_table.shape) - 1)))
print(f"Five tenure bands: chi2 = {chi2:.1f}, dof = {dof}, p = {p:.1e}, Cramer's V = {cramers_v:.3f}")

**Result**
- All three tests reject H0 (p < 0.001), but in **opposite directions**: new partners are less risky before May and far riskier after.
- Cramér's V = 0.08 is a **small** effect. Tenure alone explains little of the fraud.

## 4. Logistic Regression: Tenure as a Continuous Variable

**Question:** Does fraud risk fall as partner tenure rises?

$$\text{logit}\,P(\text{fraud}) = \beta_0 + \beta_1 \cdot \text{tenure\_years}$$

A **negative, significant β₁** means longer tenure goes with lower fraud odds. Standard errors are **clustered by partner**, because repeated claims from one partner are not independent.

In [ ]:
m_all = logit_clustered("is_fraud ~ tenure_years", claims)
m_pre = logit_clustered("is_fraud ~ tenure_years", pre)
m_post = logit_clustered("is_fraud ~ tenure_years", post)

pd.concat([
    coef_table(m_all, "All data", ["tenure_years"]),
    coef_table(m_pre, "Before May 2026", ["tenure_years"]),
    coef_table(m_post, "After May 2026", ["tenure_years"]),
]).set_index("Model")

**Check with controls:** one model with a tenure × period interaction, controlling for partner type and claim amount.

In [ ]:
m_full = logit_clustered("is_fraud ~ tenure_years * post_may + C(partner_type) + np.log(claim_amount_inr)", claims)
terms = ["tenure_years", "post_may", "tenure_years:post_may",
         "C(partner_type)[T.franchise]", "C(partner_type)[T.freelance_technician]", "np.log(claim_amount_inr)"]
coef_table(m_full, "Interaction", terms).drop(columns="Model")

In [ ]:
grid = pd.DataFrame({"tenure_years": np.linspace(0, 4, 81)})
fig, ax = plt.subplots(figsize=(8, 4))
for label, model, df in [("Before May 2026", m_pre, pre), ("After May 2026", m_post, post)]:
    line, = ax.plot(grid["tenure_years"], model.predict(grid) * 100, label=label)
    obs = df.groupby(pd.cut(df["tenure_years"], np.arange(0, 4.5, 0.5)), observed=True)["is_fraud"].mean() * 100
    ax.scatter([iv.mid for iv in obs.index], obs.values, color=line.get_color(), alpha=.6)
ax.set(title="Fitted fraud probability vs tenure (dots = observed)", xlabel="Partner tenure (years)",
       ylabel="Fraud probability (%)")
ax.legend(); plt.tight_layout(); plt.show()

**Result**
- **All data:** β₁ = −0.04, p = 0.82. No relationship between tenure and fraud.
- **Before May:** β₁ is **positive**: longer-tenured partners were slightly riskier.
- **After May:** β₁ = −1.52, p < 0.001. Each extra year of tenure cuts fraud odds by about 78%.
- With controls, the tenure × period interaction stays significant: the tenure effect **flips** after the rule change.

## 5. Partner-Level Test

**Question:** Are new partners, *as a group*, more likely to commit fraud?
Here each partner counts once (did they have any fraud claim?), so one heavy fraudster cannot dominate the result.

In [ ]:
partner_level = (claims.groupby("partner_id")["is_fraud"].agg(claims="size", frauds="sum")
                        .join(partners.set_index("partner_id")["new_cohort"]))
partner_level["any_fraud"] = (partner_level["frauds"] > 0).astype(int)

summary = (partner_level.groupby("new_cohort")
           .agg(partners=("any_fraud", "size"), with_fraud=("any_fraud", "sum"), frauds=("frauds", "sum"))
           .rename(index={1: "New cohort (since Jul 2025)", 0: "Established"}))
summary["% with fraud"] = (summary["with_fraud"] / summary["partners"] * 100).round(1)

table = pd.crosstab(partner_level["new_cohort"], partner_level["any_fraud"])
chi2, p, _, _ = stats.chi2_contingency(table)
print(f"Chi2 = {chi2:.2f}, p = {p:.2f} | Fisher p = {stats.fisher_exact(table)[1]:.2f}")
summary

**Result**
- 17.4% of new partners vs 13.4% of established partners had any fraud. The difference is **not significant** (p = 0.62).
- New partners are **not** more likely to be fraudsters as a group.

## 6. Concentration: A Few Outlets or All New Partners?

**Question:** Is the post-May new-partner effect driven by a handful of partners?

In [ ]:
by_partner = (claims[claims["new_partner"] == 1].groupby("partner_id")["is_fraud"]
               .agg(claims="size", frauds="sum").sort_values("frauds", ascending=False))
top5 = by_partner.head(5)
print(f"Top 5 partners: {top5['frauds'].sum()} of {by_partner['frauds'].sum()} new-partner frauds")
top5

In [ ]:
without_top5 = claims[~claims["partner_id"].isin(top5.index)]
pd.DataFrame([
    chi_square_2x2(without_top5, "All data, top 5 removed"),
    chi_square_2x2(without_top5[without_top5["post_may"] == 1], "After May, top 5 removed"),
]).set_index("Test")

**Result**
- **5 partners**, all first-year outlets, hold 29 of 35 new-partner frauds.
- Without them, new partners' fraud rate over all data is **0.41%**, *lower* than established partners (1.11%).
- After May a smaller gap remains, but it rests on only 5 fraud cases (min expected count < 5, so read Fisher's p).

## 7. Robustness Check: Open Investigations

**Question:** Could dropping the 202 open (blank-label) claims bias the comparison?

In [ ]:
(all_claims.assign(open_case=all_claims["is_fraud"].isna())
           .groupby(["period", "group"])["open_case"].agg(claims="size", open_cases="sum", rate="mean")
           .assign(**{"open_%": lambda t: (t.pop("rate") * 100).round(1)})
           .reindex(["Before May 2026", "After May 2026"], level=0))

**Result:** open cases are slightly *rarer* among new partners (1.9% vs 3.0% after May). Dropping them does not hide new-partner fraud.

## 8. Conclusion

| Question | Answer |
|---|---|
| Are new partners the problem overall? | **No.** Tenure does not predict fraud over the full period |
| Before May 2026? | **No.** New partners were cleaner than established ones |
| After May 2026? | **Yes, at claim level.** But driven by **5 outlets** |
| Are new partners riskier as a group? | **No.** 38 of 46 new partners have no fraud |

### Recommendations
1. **Target outlets, not the cohort.** Review claims from partners with recent confirmed fraud and unusual claim patterns.
2. **Do not flag all new partners.** It would fill the 40-claims-a-month review desk with genuine customers, at Rs 380 goodwill each.
3. **For the fraud model:** use partner-specific signals (own recent fraud history, claim behaviour) together with the post-May regime, not a blanket tenure flag.

### Limitations
- Only ~2 months and 36 frauds after May, so those estimates have wide confidence intervals.
- Association, not cause: new partners and the rule change arrived at the same time.
- Some legacy Zoho "0" labels may be open cases; this affects only the before-May period.